# Level 2 On-Demand Evaluation

Invokes the deployed agent on Bedrock AgentCore Runtime and scores the resulting OTEL spans.

## Pre-flight findings

| Check | Result |
|---|---|
| Auth type | **IAM SigV4** (`RuntimeAuthorizerConfiguration.usingIAM()`) |
| Runtime name | `sample_mlops_agent` |
| Runtime ARN | CloudFormation output `AgentRuntimeArn` on stack `sample-mlops-agent-agentcore` |
| Agent framework | **Claude Agent SDK** (`claude-agent-sdk`) |
| OTEL traces | X-Ray via `https://xray.{region}.amazonaws.com/v1/traces` -> `aws/spans` |
| OTEL logs | `otlp` -> CloudWatch Logs (runtime log group) |
| `AGENT_OBSERVABILITY_ENABLED` | `true` |
| `OTEL_TRACES_SAMPLER` | `always_on` |
| `OTEL_LOGS_EXPORTER` | `otlp` (LLO events via ADOT LLOHandler) |
| `OTEL_PYTHON_LOGGING_AUTO_INSTRUMENTATION_ENABLED` | `true` (LoggingHandler removed at agent startup) |
| `OTEL_SEMCONV_STABILITY_OPT_IN` | `gen_ai_tool_definitions` |
| `OTEL_ATTRIBUTE_VALUE_LENGTH_LIMIT` | `4096` |
| `cloud.resource_id` in `OTEL_RESOURCE_ATTRIBUTES` | patched via `UpdateAgentRuntime` |
| Agent span name | `invoke_agent sample_mlops_agent` |
| Agent span `gen_ai.system` | `strands-agents` |
| Agent span `gen_ai.operation.name` | `invoke_agent` |

## How evaluation works

1. **Span export**: `opentelemetry-instrument` auto-instruments the agent. Spans flow via
   X-Ray OTLP to `aws/spans`. Custom span events (`gen_ai.user.message`, `gen_ai.choice`)
   are stripped by X-Ray during forwarding.

2. **LLO event export**: The ADOT LLOHandler intercepts span events before export,
   converts them to structured log records (input/output messages in Bedrock content block
   format), and exports them via the OTLP logs endpoint to the runtime log group.  These
   LLO events are what the evaluate API uses to extract user queries and agent responses.

3. **On-demand evaluation** (this notebook): Cell 6 queries both `aws/spans` and the
   runtime log group, combines spans + LLO events, and calls the evaluate API directly.

4. **Online evaluation**: Bedrock AgentCore monitors the runtime log group continuously
   and evaluates sampled interactions automatically.  No client-side processing needed.

In [ ]:
# Cell 1 — Install (run once)
%pip install -q \
    "bedrock-agentcore-starter-toolkit" \
    boto3 "botocore[crt]" python-dotenv pandas --upgrade

In [ ]:
# Cell 2 — Imports and AWS session
import json
import os
import time
import uuid
from datetime import datetime

import boto3
import pandas as pd
from botocore.config import Config
from botocore.exceptions import ClientError
from dotenv import load_dotenv
from IPython.display import display

load_dotenv(dotenv_path=".env", override=False)

_config = Config(retries={"max_attempts": 10, "mode": "standard"})
session = boto3.Session(profile_name=os.environ.get("AWS_PROFILE", "default"))
AWS_REGION = session.region_name or "us-east-1"

# Stack that contains the AgentCore Runtime
AGENTCORE_STACK = os.environ.get("AGENTCORE_STACK", "sample-mlops-agent-agentcore")

_identity = session.client("sts", region_name=AWS_REGION, config=_config).get_caller_identity()
print(f"✓ AWS Credentials")
print(f"  Profile : {os.environ.get('AWS_PROFILE', 'default')}")
print(f"  Account : ...{_identity['Account'][-4:]}")
print(f"  Role    : {_identity['Arn'].split('/')[-1]}")
print(f"  Region  : {AWS_REGION}")

In [ ]:
# Cell 3 — CDK config lookup: IAM SigV4 pattern
# Runtime ARN is published as CloudFormation output 'AgentRuntimeArn' on the agentcore stack.

_cfn = session.client("cloudformation", region_name=AWS_REGION, config=_config)

_all_stacks: list[dict] = []
for _page in _cfn.get_paginator("describe_stacks").paginate():
    _all_stacks.extend(_page.get("Stacks", []))

_target = next(
    (s for s in _all_stacks if AGENTCORE_STACK in s["StackName"]),
    None,
)
if not _target:
    available = [s["StackName"] for s in _all_stacks if "mlops" in s["StackName"].lower()]
    raise RuntimeError(
        f"Stack '{AGENTCORE_STACK}' not found.\n"
        f"Candidates: {available}\n"
        f"Deploy first: cd cdk && npx cdk deploy sample-mlops-agent-agentcore"
    )

_outputs = {o["OutputKey"]: o["OutputValue"] for o in _target.get("Outputs", [])}
print(f"Stack  : {_target['StackName']}")
print(f"Status : {_target['StackStatus']}")
print(f"Outputs: {list(_outputs.keys())}")

runtime_arn: str = _outputs["AgentRuntimeArn"]
agent_id: str = runtime_arn.rsplit("/", 1)[-1]

print(f"\nruntime_arn = .../{agent_id}")
print(f"agent_id    = {agent_id}")

In [ ]:
# Cell 4 — Invoke agent via IAM SigV4
#
# Payload schema (from agent/main.py):
#   session_id  : str  — AgentCore session ID (pass same ID to resume a session)
#   messages    : list — [{"role": "user", "content": "..."}]  (preferred)
#   message     : str  — fallback when messages is absent
#
# Response: BedrockAgentCoreApp streams events in SSE format — each event is:
#   data: {json}\n\n
# Event types: RUN_STARTED, TEXT_MESSAGE_CHUNK, TOOL_CALL_CHUNK, RUN_FINISHED

TEST_INPUTS = [
    {
        "label": "list jobs",
        "message": "List my recent SageMaker training jobs and their current status.",
    },
    {
        "label": "mlflow query",
        "message": "Show me the MLflow experiments and their latest run metrics.",
    },
]

_ac = session.client("bedrock-agentcore", region_name=AWS_REGION, config=_config)


def _read_streaming_response(resp: dict) -> tuple[str, list[dict]]:
    """Read all streamed events from an invoke_agent_runtime response.

    BedrockAgentCoreApp._convert_to_sse() wraps every yielded dict as:
        data: {json}\\n\\n
    Each line starting with "data: " carries a JSON payload; all other lines
    (blank separators, etc.) are ignored.

    Args:
        resp: Raw boto3 response dict from invoke_agent_runtime.

    Returns:
        text_output: Concatenated TEXT_MESSAGE_CHUNK deltas.
        events: List of all parsed event dicts.
    """
    raw_bytes = b""
    response_body = resp.get("response", b"")
    if hasattr(response_body, "iter_chunks"):
        for chunk in response_body.iter_chunks():
            raw_bytes += chunk
    elif hasattr(response_body, "read"):
        raw_bytes = response_body.read()
    else:
        for chunk in response_body:
            if isinstance(chunk, bytes):
                raw_bytes += chunk
            elif isinstance(chunk, dict):
                raw_bytes += chunk.get("chunk", {}).get("bytes", b"")

    # Only parse lines that carry a JSON payload (strip the 6-byte "data: " prefix).
    events: list[dict] = [
        json.loads(l[6:]) for l in raw_bytes.splitlines() if l.startswith(b"data: ")
    ]

    text_output = "".join(
        e.get("delta", "") for e in events if e.get("type") == "TEXT_MESSAGE_CHUNK"
    )
    return text_output, events


session_ids: list[str] = []
responses: list[dict] = []

for i, test in enumerate(TEST_INPUTS):
    sid = str(uuid.uuid4())
    session_ids.append(sid)

    payload = json.dumps(
        {
            "session_id": sid,
            "messages": [{"role": "user", "content": test["message"]}],
        }
    ).encode("utf-8")

    start = time.time()
    try:
        resp = _ac.invoke_agent_runtime(
            agentRuntimeArn=runtime_arn,
            runtimeSessionId=sid,
            payload=payload,
            qualifier="DEFAULT",
        )
    except ClientError as exc:
        code = exc.response["Error"]["Code"]
        msg  = exc.response["Error"]["Message"]
        raise RuntimeError(f"[{i}] Invocation failed ({code}): {msg}") from exc

    text, events = _read_streaming_response(resp)
    elapsed = time.time() - start
    responses.append({"label": test["label"], "session_id": sid, "text": text, "events": events})

    print(f"[{i}] {test['label']} — {elapsed:.1f}s — session: {sid[:8]}...")
    print(f"     events: {[e.get('type') for e in events]}")
    print(f"     text  : {text[:120]}..." if len(text) > 120 else f"     text  : {text}")

print(f"\n✓ {len(TEST_INPUTS)} invocation(s) complete")

In [ ]:
# Cell 5 — Wait for OTEL spans to be indexed in aws/spans

_logs = session.client("logs", region_name=AWS_REGION, config=_config)
_WAIT_S = 90

print(f"Sleeping {_WAIT_S}s for CWL Insights span indexing...")
for i in range(_WAIT_S // 10):
    time.sleep(10)
    print(f"  [{(i+1)*10}s] ...")

# Confirm spans are visible per session
_confirmed: list[bool] = []
for sid in session_ids:
    ok = False
    for attempt in range(4):
        try:
            qr = _logs.start_query(
                logGroupName="aws/spans",
                startTime=int(time.time()) - 600,
                endTime=int(time.time()),
                queryString=(
                    f"fields spanId "
                    f"| filter attributes.session.id = '{sid}' "
                    f"| filter ispresent(attributes.gen_ai.operation.name) "
                    f"| limit 1"
                ),
            )
            for _ in range(20):
                time.sleep(1)
                r = _logs.get_query_results(queryId=qr["queryId"])
                if r["status"] == "Complete":
                    if r.get("results"):
                        ok = True
                    break
        except Exception as e:
            print(f"  [{sid[:8]}] query error: {e}")
        if ok:
            break
        if attempt < 3:
            print(f"  [{sid[:8]}] not yet visible (attempt {attempt+1}/4), waiting 15s...")
            time.sleep(15)

    _confirmed.append(ok)
    status = "indexed" if ok else "NOT FOUND (check cloud.resource_id patch)"
    print(f"  {'ok' if ok else '!!'} [{sid[:8]}...] {status}")

print(f"\n{sum(_confirmed)}/{len(session_ids)} session(s) confirmed in aws/spans")

In [ ]:
# Cell 6 — Evaluate sessions
#
# The evaluate API requires:
#   1. An invoke_agent span (gen_ai.operation.name = "invoke_agent",
#      gen_ai.system = "strands-agents").
#   2. A corresponding LLO event (log record) with body in LLOHandler format:
#      input.messages[].content = {"content": <json-serialized content blocks>}
#      output.messages[].content = {"message": <text>, "finish_reason": "end_turn"}
#
# With OTEL_LOGS_EXPORTER=otlp enabled in the CDK stack, the ADOT LLOHandler
# natively exports LLO events to the runtime log group.

from datetime import datetime, timedelta, timezone

from bedrock_agentcore.evaluation.utils.cloudwatch_span_helper import CloudWatchSpanHelper

_dp = session.client("bedrock-agentcore", region_name=AWS_REGION, config=_config)
_cp = session.client("bedrock-agentcore-control", region_name=AWS_REGION, config=_config)
_cw = CloudWatchSpanHelper(region=AWS_REGION)
_runtime_lg = f"/aws/bedrock-agentcore/runtimes/{agent_id}-DEFAULT"
_qend = datetime.now(timezone.utc)
_qstart = _qend - timedelta(days=7)

# ── helpers ──────────────────────────────────────────────────────────────────

_level_cache: dict[str, str] = {}

def _get_level(eid: str) -> str:
    """Look up evaluator level (SESSION, TRACE, or TOOL_CALL)."""
    if eid not in _level_cache:
        try:
            _level_cache[eid] = _cp.get_evaluator(evaluatorId=eid)["level"]
        except Exception:
            _level_cache[eid] = "SESSION"
    return _level_cache[eid]


def _collect(sid: str) -> list[dict]:
    """Collect spans from aws/spans and native LLO events from the runtime log group.

    Returns the combined list sorted by timestamp, ready for the evaluate API.
    """
    # aws/spans: use attributes.session.id (CWL resolves the nested path
    # correctly in the span schema)
    span_q = (
        f"fields @timestamp, @message"
        f'\n| filter attributes.session.id = "{sid}"'
        f"\n| filter ispresent(scope.name)"
        f"\n| filter ispresent(traceId)"
        f"\n| filter ispresent(spanId)"
        f"\n| sort @timestamp asc"
    )
    # Runtime log group: use @message substring match because CWL Insights
    # cannot resolve dotted attribute keys like attributes.session.id when
    # "session.id" is a single key name (not a nested path).
    event_q = (
        f"fields @timestamp, @message"
        f'\n| filter @message like "{sid}"'
        f"\n| filter ispresent(scope.name)"
        f"\n| filter ispresent(traceId)"
        f"\n| filter ispresent(spanId)"
        f"\n| sort @timestamp asc"
    )
    raw_spans = _cw.query_log_group("aws/spans", sid, _qstart, _qend, query_string=span_q)
    cw_events = _cw.query_log_group(_runtime_lg, sid, _qstart, _qend, query_string=event_q)

    docs = raw_spans + cw_events
    docs.sort(key=lambda d: d.get("endTimeUnixNano", d.get("timeUnixNano", 0)))

    print(f"  {len(raw_spans)} spans + {len(cw_events)} LLO events")
    return docs


# ── run evaluation ───────────────────────────────────────────────────────────

_EVALUATORS = [
    "Builtin.GoalSuccessRate",
    "Builtin.Correctness",
    "Builtin.ToolSelectionAccuracy",
    "Builtin.ToolParameterAccuracy",
]

print(f"Ready  (region={AWS_REGION}, agent_id={agent_id})")

all_results: list[dict] = []
for idx, sid in enumerate(session_ids):
    label = responses[idx]["label"]
    print(f"\n[{idx}] Evaluating session {sid[:8]}... ({label})")
    docs = _collect(sid)
    n_spans = sum(1 for d in docs if "name" in d and "kind" in d)
    n_events = sum(1 for d in docs if "body" in d)
    print(f"  docs: {len(docs)} ({n_spans} spans + {n_events} events)")

    if not docs:
        all_results.append({"idx": idx, "label": label, "session": sid[:8],
                            "evaluator": "ERROR", "score": None, "label_str": None,
                            "explanation": "No spans found"})
        continue

    for ev_id in _EVALUATORS:
        level = _get_level(ev_id)
        req: dict = {"evaluationInput": {"sessionSpans": docs}}

        if level == "TRACE":
            tids = list(dict.fromkeys(d.get("traceId") for d in docs if d.get("traceId")))
            req["evaluationTarget"] = {"traceIds": tids[:10]}
        elif level == "TOOL_CALL":
            tsids = [
                d["spanId"] for d in docs
                if isinstance(d.get("attributes"), dict)
                and d["attributes"].get("gen_ai.operation.name") == "execute_tool"
                and d.get("spanId")
            ]
            if not tsids:
                print(f"  skip {ev_id}: no tool spans")
                continue
            req["evaluationTarget"] = {"spanIds": tsids[:10]}

        try:
            resp = _dp.evaluate(evaluatorId=ev_id, **req)
            for r in resp.get("evaluationResults", []):
                all_results.append({
                    "idx": idx, "label": label, "session": sid[:8],
                    "evaluator": r.get("evaluatorId", ev_id),
                    "score": r.get("value"),
                    "label_str": r.get("label"),
                    "explanation": str(r.get("explanation", ""))[:140],
                })
            print(f"  {ev_id}: {len(resp.get('evaluationResults', []))} result(s)")
        except Exception as exc:
            print(f"  {ev_id}: {exc}")
            all_results.append({
                "idx": idx, "label": label, "session": sid[:8],
                "evaluator": ev_id, "score": None, "label_str": None,
                "explanation": str(exc)[:140],
            })

print(f"\nTotal: {len(all_results)} result(s)")

In [ ]:
# Cell 7 — Results summary

df = pd.DataFrame(all_results)
print("=== ALL RESULTS ===")
display(df)

scores = [r["score"] for r in all_results if r["score"] is not None]
if scores:
    print(f"\nOverall mean score : {sum(scores)/len(scores):.3f}")
    by_eval = df[df["score"].notna()].groupby("evaluator")["score"].mean()
    print("\nMean score by evaluator:")
    display(by_eval.to_frame())
else:
    print("\n⚠️  No scores returned — check the errors above.")

# Store for downstream notebooks
ondemand_session_ids = session_ids
ondemand_agent_id = agent_id
%store ondemand_session_ids
%store ondemand_agent_id
print(f"\n✓ Stored ondemand_agent_id and ondemand_session_ids for downstream notebooks")

## Infrastructure status

| Item | Status |
|---|---|
| `OTEL_ATTRIBUTE_VALUE_LENGTH_LIMIT: '4096'` | deployed |
| `OTEL_LOGS_EXPORTER: 'otlp'` | deployed — LLO events flow to runtime log group |
| `OTEL_PYTHON_LOGGING_AUTO_INSTRUMENTATION_ENABLED: 'true'` | deployed — LoggingHandler removed at agent startup |
| `OTEL_EXPORTER_OTLP_LOGS_HEADERS` | patched by PatchRuntimeOtel to target runtime log group |
| `PatchRuntimeOtel` Custom Resource | deployed — patches `cloud.resource_id` + log group headers + creates log stream |
| `agent/main.py` span name | `invoke_agent sample_mlops_agent` |
| `gen_ai.system` on agent span | `strands-agents` |
| `gen_ai.operation.name` on agent span | `invoke_agent` |
| `gen_ai.request.model` on agent span | `anthropic.claude-opus-4-5-20251001-v1:0` |
| LLO events in runtime log group | native via ADOT LLOHandler |
| CWL query for runtime LG events | `@message like` (not `attributes.session.id` — dotted key) |

## UI-driven evaluation (Evals tab)

This on-demand notebook flow is now also exposed in the **Evals tab** of the app,
which drives the *same* AgentCore evaluation pipeline:

- **Online metrics** — an `OnlineEvaluationConfig` (CDK stack `sample-mlops-agent-evals`)
  continuously scores live sessions with `Builtin.GoalSuccessRate`, `Builtin.Helpfulness`,
  `Builtin.Correctness`, and the custom **confirmation-gate** evaluator. Scores land in the
  CloudWatch namespace `Bedrock-AgentCore/Evaluations` (see notebook 1).
- **Batch eval** — the tab's *Run batch evaluation* button calls `StartBatchEvaluation`
  over recent sessions (data-plane `bedrock-agentcore` client), the same scoring this
  notebook performs on demand.
- **Optimization** — *Generate recommendation* calls `StartRecommendation` for an improved
  system prompt (scored against `GoalSuccessRate`) or tool descriptions; *Apply* writes the
  new prompt to `s3://<session-bucket>/skills/system-prompt.md`, which the agent hydrates on
  its next turn (no redeploy).

### Custom confirmation-gate evaluator

`lambda/eval_confirmation_gate` is a SESSION-level code-based evaluator: it inspects the
session's `execute_tool <name>` spans and scores **AGREED / DISAGREED / ABSTAIN** on whether a
confirmation prompt ("Proceed? (yes/no)") preceded any billable tool call
(`submit_training_job` / `deploy_model`) — the MLOps guardrail from the SageMaker skill.


In [ ]:
# List the registered evaluators + online-eval config for this account.
cp = session.client('bedrock-agentcore-control', region_name=AWS_REGION, config=_config)
try:
    evs = cp.list_evaluators().get('evaluators', [])
    print('evaluators:', [e.get('evaluatorName') or e.get('evaluatorId') for e in evs])
    cfgs = cp.list_online_evaluation_configs().get('onlineEvaluationConfigs', [])
    print('online configs:', [(c.get('onlineEvaluationConfigName'), c.get('executionStatus')) for c in cfgs])
except Exception as e:
    print('list failed (check region/permissions):', type(e).__name__, e)
